[Back to NLP guideline](Natural-Language-Processing.html)


## **Tokenisation**

Tokenisation is the process of breaking raw text into smaller units called **tokens**. These tokens are then mapped to token IDs and embeddings before being passed into an NLP model.

```text
Raw text ? Tokens ? Token IDs ? Embeddings ? Model
```

A token can be a word, a character, a subword, punctuation, or a special symbol such as `[CLS]`, `[SEP]`, `<pad>`, or `<unk>`.

Tokenisation looks like a small preprocessing step, but it strongly affects:

- vocabulary size
- sequence length
- unknown-word handling
- multilingual coverage
- model efficiency
- token-level evaluation such as perplexity

![Tokenisation levels](assets/tokenisation-levels.svg)

### **Basic Tokenisation Methods**

#### **Whitespace Tokenisation**

The simplest method is to split text on whitespace.

```text
I love natural language processing
? [I, love, natural, language, processing]
```

This is a reasonable baseline for simple English text, but it has many edge cases:

- Some languages do not use spaces between words.
- Punctuation stays attached to words.
- Compound words may become rare huge tokens.
- Contractions such as `don't` are ambiguous.
- New names, typos, and domain terms can become unknown words.

<details>
<summary>Python Whitespace Tokenisation</summary>

```python
text = "I love natural language processing."

tokens = text.split()
print(tokens)

# Output: ['I', 'love', 'natural', 'language', 'processing.']
# Problem: punctuation remains attached to the final word.
```
</details>

#### **Rule-based Tokenisation**

Rule-based tokenisers use handcrafted rules or regular expressions to handle punctuation, contractions, numbers, and other special cases.

```text
I love NLP.
? [I, love, NLP, .]
```

Rule-based tokenisation is more controlled than whitespace splitting, but the rules are usually language-specific and can be brittle.

<details>
<summary>Python Regex Tokenisation</summary>

```python
import re

text = "Don't split U.S.A. too badly!"

# Simple rule: match words or individual punctuation symbols
pattern = r"\w+|[^\w\s]"
tokens = re.findall(pattern, text)

print(tokens)
```
</details>

#### **Character-level Tokenisation**

Character-level tokenisation splits text into individual characters.

```text
token ? [t, o, k, e, n]
```

**Pros**:

- Very small vocabulary.
- No true unknown-word problem.
- Can handle rare words and typos.

**Cons**:

- Creates long sequences.
- Individual characters have weak semantic meaning.
- Longer sequences make modelling more expensive.

<details>
<summary>Python Character Tokenisation</summary>

```python
text = "tokenisation"

characters = list(text)
print(characters)
```
</details>

#### **Word-level Tokenisation**

Word-level tokenisation treats each word as a token.

```text
I like chocolate
? [I, like, chocolate]
```

**Pros**:

- Intuitive and readable.
- Shorter sequences than character-level tokenisation.
- Works well when the vocabulary covers the domain.

**Cons**:

- Vocabulary can become huge.
- Rare and unseen words cause OOV problems.
- Related forms may be treated as unrelated tokens.

### **Why Subword Tokenisation?**

Subword tokenisation is a compromise between word-level and character-level tokenisation.

```text
unhappiness ? [un, happiness]
tokenization ? [token, ization]
lowest ? [low, est]
```

It is useful because it can represent rare or unseen words using smaller known pieces, while keeping sequences shorter than pure character-level tokenisation.

| Method | Vocabulary Size | Sequence Length | Handles Rare Words | Main Weakness |
|---|---:|---:|---|---|
| Character-level | Small | Long | Yes | Harder to model semantics |
| Word-level | Large | Short | Poorly | OOV problem |
| Subword-level | Medium | Medium | Yes | Token boundaries may be unintuitive |

### **Byte Pair Encoding (BPE)**

Byte Pair Encoding is a common subword tokenisation algorithm. It starts with small units and repeatedly merges frequent adjacent pairs into larger units.

The algorithm is:

1. Start with a vocabulary of individual characters.
2. Count adjacent symbol pairs in the training data.
3. Merge the most frequent adjacent pair into a new vocabulary item.
4. Repeat until the vocabulary reaches the desired size.

![BPE merge process](assets/bpe-merge-process.svg)

Example:

```text
low      ? l o w _
lowest   ? l o w e s t _
newer    ? n e w e r _
```

If `l + o` is frequent, BPE may merge it into `lo`. Later, `lo + w` may become `low`, and `low + e` may become `lowe`.

After training, BPE tokenises new input by applying the learned merge rules in order.

**Strengths**:

- Good open-vocabulary behavior.
- Learns subwords from data.
- Balances vocabulary size and sequence length.
- Strong baseline for modern NLP systems.

**Weaknesses**:

- Pair frequency is not always linguistically meaningful.
- Token boundaries can be unintuitive.
- Different training corpora produce different merge rules.

<details>
<summary>Python Simple BPE Training Example</summary>

```python
from collections import defaultdict

corpus = {
    tuple("low_"): 5,
    tuple("lowest_"): 2,
    tuple("newer_"): 6,
    tuple("wider_"): 3,
    tuple("new_"): 2
}

def get_pair_counts(vocab):
    counts = defaultdict(int)
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            counts[(symbols[i], symbols[i + 1])] += freq
    return counts

def merge_pair(pair, vocab):
    merged_vocab = {}
    merged_symbol = "".join(pair)

    for symbols, freq in vocab.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        merged_vocab[tuple(new_symbols)] = freq

    return merged_vocab

vocab = corpus
merge_rules = []

for step in range(8):
    pair_counts = get_pair_counts(vocab)
    best_pair = max(pair_counts, key=pair_counts.get)
    merge_rules.append(best_pair)
    vocab = merge_pair(best_pair, vocab)

    print(f"Step {step + 1}: merge {best_pair} -> {''.join(best_pair)}")
    print(vocab)
```
</details>

### **WordPiece**

WordPiece is a subword tokenisation family used by BERT-style models. Like BPE, it builds a vocabulary from smaller pieces. The important difference is **how the next piece is chosen during training**.

There are two related ideas that are easy to mix together:

| Version | Main idea | Typical explanation |
|---|---|---|
| **Standard WordPiece** | Add the piece that improves the language-model likelihood the most | Likelihood / perplexity objective |
| **HuggingFace WordPiece** | Add the pair with the highest association score | `freq(ab) / (freq(a) * freq(b))` |

Both versions normally use the same inference-time rule: **longest-match-first** tokenisation with continuation markers such as `##`.

#### **Standard WordPiece**

The original WordPiece idea is not just "merge the most frequent pair". It asks a more model-based question:

> If this new subword piece is added to the vocabulary, how much better can the corpus be represented under the tokenisation model?

A simplified training flow is:

| Step | Operation | Intuition |
|---:|---|---|
| 1 | Start with a small vocabulary | Usually characters plus special tokens |
| 2 | Mark inside-word pieces | For BERT-style vocabularies, continuation pieces use `##` |
| 3 | Generate candidate new pieces | Usually adjacent pieces inside words |
| 4 | Score each candidate | Estimate improvement to corpus likelihood / perplexity |
| 5 | Add the best candidate | The vocabulary grows by one piece |
| 6 | Repeat | Stop at the target vocabulary size |

So Standard WordPiece is best understood as a **likelihood-improving vocabulary construction algorithm**. It prefers pieces that make the tokenised corpus easier to model, not necessarily pieces with the highest raw frequency.

```text
Initial pieces:
[u, ##n, ##a, ##f, ##f, ##a, ##b, ##l, ##e]

Possible learned pieces over time:
[u, ##n, ##aff, ##able, unaffable]
```

At inference time, after the vocabulary has been trained, the tokeniser does not re-run this training objective. It simply segments each new word using the existing vocabulary.

<details>
<summary>Python Greedy WordPiece Tokenisation</summary>

```python
def wordpiece_tokenize(word, vocab, unk_token="[UNK]"):
    tokens = []
    start = 0

    while start < len(word):
        end = len(word)
        matched = None

        while start < end:
            piece = word[start:end]
            if start > 0:
                piece = "##" + piece

            if piece in vocab:
                matched = piece
                break

            end -= 1

        if matched is None:
            return [unk_token]

        tokens.append(matched)
        start = end

    return tokens

vocab = {
    "[UNK]",
    "un", "##aff", "##able",
    "token", "##ization",
    "play", "##ing"
}

for word in ["unaffable", "tokenization", "playing", "unknownword"]:
    print(word, "->", wordpiece_tokenize(word, vocab))
```
</details>

#### **HuggingFace WordPiece**

The HuggingFace course explanation, which is also the version commonly shown in teaching material, presents WordPiece with a concrete pair scoring formula:

$$
score(a,b) = \frac{freq(ab)}{freq(a) \times freq(b)}
$$

This version still grows a vocabulary by merging adjacent pieces, but it chooses the pair with the highest **association score** rather than the highest raw frequency.

The denominator matters. If `a` and `b` are individually very common, then a high pair frequency may not be very informative. But if two rare pieces often appear together, the score becomes high because the pair is strongly associated.

| Pair | Pair Frequency | First Frequency | Second Frequency | HuggingFace WordPiece Score |
|---|---:|---:|---:|---:|
| `u + n` | 20 | 100 | 80 | 0.0025 |
| `q + z` | 5 | 6 | 7 | 0.1190 |

BPE would choose `u + n` because it has higher raw frequency. HuggingFace-style WordPiece may choose `q + z` because the two pieces are much more strongly tied together.

A simplified HuggingFace-style training flow is:

| Step | Operation |
|---:|---|
| 1 | Split words into initial character pieces |
| 2 | Count individual piece frequencies |
| 3 | Count adjacent pair frequencies |
| 4 | Compute `freq(ab) / (freq(a) * freq(b))` for every pair |
| 5 | Merge the highest-scoring pair |
| 6 | Repeat until the vocabulary reaches the target size |

<details>
<summary>Python HuggingFace-style WordPiece Pair Scoring</summary>

```python
from collections import defaultdict

corpus = {
    ("u", "##n", "##a", "##f", "##f", "##a", "##b", "##l", "##e"): 10,
    ("u", "##n", "##h", "##a", "##p", "##p", "##y"): 8,
    ("q", "##z"): 5,
}

def get_wordpiece_scores(corpus):
    token_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for tokens, word_freq in corpus.items():
        for token in tokens:
            token_freqs[token] += word_freq
        for pair in zip(tokens, tokens[1:]):
            pair_freqs[pair] += word_freq

    scores = {}
    for pair, pair_freq in pair_freqs.items():
        left, right = pair
        scores[pair] = pair_freq / (token_freqs[left] * token_freqs[right])

    return pair_freqs, token_freqs, scores

pair_freqs, token_freqs, scores = get_wordpiece_scores(corpus)

for pair, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(pair, "pair_freq=", pair_freqs[pair], "score=", round(score, 4))
```
</details>

#### **Continuation Marker: `##`**

BERT-style WordPiece vocabularies often mark continuation pieces with `##`.

```text
unaffable -> [un, ##aff, ##able]
playing   -> [play, ##ing]
```

This marker tells the model whether a piece begins a word or continues a previous piece.

| Token | Meaning |
|---|---|
| `play` | Can start a word |
| `##ing` | Must continue a word |
| `token` | Can start a word |
| `##ization` | Must continue a word |

#### **Longest-match-first Tokenisation**

Once the vocabulary is trained, WordPiece usually tokenises new words greedily using **longest-match-first**.

For example, if the vocabulary contains:

```text
un, ##aff, ##able, ##a, ##ble
```

Then:

```text
unaffable -> un + ##aff + ##able
```

The tokeniser first searches for the longest vocabulary item starting at the current position. If no valid piece can be found, the whole word often becomes `[UNK]` in classical BERT-style WordPiece.

#### **BPE vs Standard WordPiece vs HuggingFace WordPiece**

| Method | Training decision | Main intuition |
|---|---|---|
| **BPE** | Merge the most frequent adjacent pair | Frequent co-occurrence is useful |
| **Standard WordPiece** | Add the piece that best improves corpus likelihood / perplexity | Better vocabulary means better modelling |
| **HuggingFace WordPiece** | Merge the pair with highest `freq(ab) / (freq(a) * freq(b))` | Strong association matters more than raw frequency |

### **Unigram / SentencePiece**

Unigram tokenisation, often associated with SentencePiece, takes the opposite direction from BPE and WordPiece.

Instead of starting small and merging upward, it starts with a large candidate vocabulary and removes less useful pieces.

High-level algorithm:

1. Start with a large vocabulary containing characters, frequent substrings, and full words.
2. Estimate how useful each piece is for explaining the corpus.
3. Remove pieces that contribute least.
4. Repeat until the vocabulary reaches the desired size.

SentencePiece is useful because it can operate directly on raw text without relying on whitespace pre-tokenisation. It often represents spaces using a visible marker such as `?`.

```text
I like NLP
? [?I, ?like, ?NLP]
```

<details>
<summary>Python SentencePiece Example</summary>

```python
# pip install sentencepiece

import sentencepiece as spm

# Train a model from a text file.
# spm.SentencePieceTrainer.train(
#     input="corpus.txt",
#     model_prefix="spm_demo",
#     vocab_size=1000,
#     model_type="unigram"
# )

sp = spm.SentencePieceProcessor(model_file="spm_demo.model")

text = "I like natural language processing."

print(sp.encode(text, out_type=str))
print(sp.encode(text, out_type=int))
```
</details>

### **Byte-level BPE**

Byte-level BPE starts from bytes rather than characters or words. Since bytes can represent any text, this gives strong coverage for rare symbols, emojis, typos, and multilingual input.

GPT-style tokenisers often use byte-level BPE or related variants.

One detail is that spaces are often encoded as part of the token. For example, GPT-2/RoBERTa-style tokenisers may use `?` to indicate that a token begins after a space.

```text
"Hello world" -> [Hello, ?world]
```

**Strengths**:

- Excellent coverage.
- No true unknown characters.
- Robust on web text and multilingual text.

**Weaknesses**:

- Tokens can be hard to read.
- Token counts can be unintuitive.

<details>
<summary>Python GPT-2 Byte-level BPE Example</summary>

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

examples = [
    "Hello world",
    "Hello   world",
    "I love pizza!"
]

for text in examples:
    print(text)
    print(tokenizer.tokenize(text))
    print(tokenizer.encode(text))
```
</details>

### **Language-specific Tokenisation**

Tokenisation is not the same for all languages.

For languages such as Chinese, Japanese, and Thai, word boundaries are not always marked by spaces. In traditional NLP pipelines, word segmentation may be a separate task.

Chinese example:

```text
?????????
? [?, ??, ??????]
```

Modern multilingual models often rely on subword or byte-level tokenisation instead of explicit word segmentation, but segmentation tools are still useful in some tasks.

<details>
<summary>Python Chinese Word Segmentation with jieba</summary>

```python
# pip install jieba

import jieba

text = "?????????"
print(list(jieba.cut(text)))
```
</details>

### **Tokenisation in Modern Models**

Modern NLP models usually use subword tokenisation.

| Model Family | Common Tokenisation Style |
|---|---|
| BERT | WordPiece |
| GPT-style models | BPE / byte-level BPE variants |
| T5 / ALBERT / XLNet variants | SentencePiece / Unigram variants |
| Llama-style models | BPE / SentencePiece variants |

Tokenisation affects model cost because it changes sequence length. In Transformers, self-attention grows roughly quadratically with sequence length, so a tokeniser that produces many tokens can make the same text more expensive to process.

Tokenisation also affects evaluation. For example, language model perplexity depends on the tokenisation used, so perplexity values from different tokenisers are not always directly comparable.

### **Special Tokens**

Many neural NLP models add special tokens to represent structure.

| Token | Meaning |
|---|---|
| `[CLS]` | Classification token, often used by BERT |
| `[SEP]` | Separator between sentences or segments |
| `[MASK]` | Masked token for masked language modeling |
| `<s>` | Start of sequence |
| `</s>` | End of sequence |
| `<pad>` | Padding token for batching |
| `<unk>` | Unknown token |

Special tokens are part of the vocabulary and receive their own embeddings.

<details>
<summary>Python Token IDs and Special Tokens</summary>

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Natural language processing is useful."
encoded = tokenizer(text)

print("Tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Token IDs:", encoded["input_ids"])
print("Special tokens:", tokenizer.special_tokens_map)
```
</details>

### **Summary**

| Tokenisation Method | Core Idea | Strength | Weakness |
|---|---|---|---|
| Whitespace | Split on spaces | Simple baseline | Fails for many languages and edge cases |
| Rule-based / Regex | Use handcrafted rules | More controlled | Language-specific and brittle |
| Character-level | Split into characters | No OOV issue | Very long sequences |
| Word-level | Split into words | Intuitive | Large vocabulary, OOV problem |
| BPE | Merge most frequent adjacent pairs | Strong subword baseline | Frequency is not always meaningful |
| WordPiece | Build subword vocabulary with likelihood-style or association-style scoring | Used in BERT-style models | Boundaries can be unintuitive |
| Unigram / SentencePiece | Start large, prune pieces | Works well from raw text | More complex training |
| Byte-level BPE | BPE over bytes | Excellent coverage | Tokens can be hard to read |

A practical rule of thumb:

- Use word-level tokenisation for simple classical baselines.
- Use BPE or WordPiece for most neural NLP models.
- Use SentencePiece or byte-level methods when multilingual coverage and raw-text robustness matter.
